## 📑 Coc Fields
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/ioApH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, regexp_replace, array, col, explode, lit, when, split
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

# Check αν το run_date που δόθηκε υπάρχει στον dexposure
if run_date:
    exists = spark.sql(f'''
                       select 1 from `{env_prefix}-cdh`.derivative.dexposure
                       where BusinessDate = '{run_date}'
                       limit 1
                       ''').collect()
    if not exists:
        run_date = ''   # δεν υπάρχει -> μηδενίζεται για να πέσει στο if not run_date

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')

som_date = spark.sql(f'''SELECT last_day(add_months('{run_date}', -1)) AS SOM''').collect()[0]['SOM']
som_date = som_date.strftime('%Y-%m-%d')


In [0]:
table_name = 'Core_CocFields'

schema = (
"BusinessDate date,"
"Case_Status string,"
"BU string,"
"Account_No string,"
"Type_Of_Issuing string,"
"Clean_Account_No string,"
"Contract string,"
"Case_Code string,"
"CIF string,"
"CUST_AFM string,"
"First_Name string,"
"Last_Name string,"
"Relation_Type string,"
"Account_First_Letter string,"
"Account_First_Letter_date date,"
"UPDATED_BY_1st_LETTER string,"
"Account_First_Reminder_Letter string,"
"Account_First_Reminder_letter_Date date,"
"Updated_by_1st_reminder_letter string,"
"Account_Second_Letter string,"
"Account_Second_Letter_date date,"
"UPDATED_BY_2nd_Letter string,"
"Account_Third_Letter string,"
"Account_Third_Letter_date date,"
"UPDATED_BY_3rd_LETTER string,"
"CoC_Outstanding_Balance_LCY decimal(19,2),"
"Account_Active_Application_for_Code string,"
"Account_Status string,"
"Account_Status_Last_Update date,"
"Letter_Conduct_Code string,"
"Letter_Date_Conduct_Code date,"
"Customer_Active_Application_for_Code string,"
"Account_Application_for_code_Decision_Completed_Ever string,"
"Account_Application_for_code_Decision_Completed_Ever_Date date,"
"Active_Appeal string,"
"Date_of_Appeal date,"
"Committee_Decision_LoV string,"
"Decision_Date date,"
"Exclude_from_CoC_Letters string,"
"Exclusion_Reason_from_CoC_Letters_LoV string,"
"Coc_Exclusion_Date date,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
# env_prefix ='qa' #IL: To be Deleted!

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    som_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### NonCorePerimeter

In [0]:
NonCorePerimeter = spark.sql(
f'''
SELECT ExposureKeyQCAlpha
     , ExposureKey
     , BusinessDate
FROM `{env_prefix}-cdh`.derivative.dexposure
WHERE 1=1
AND PortfolioName <> 'Core Retail'
AND BusinessDate <= '{run_date}'
'''
)
NonCorePerimeter.createOrReplaceTempView("NonCorePerimeter")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT DISTINCT
        CAST({som_date!r} AS DATE)                                                        AS BusinessDate
      , CAST(Case_Status AS STRING)                                                       AS Case_Status
      , CAST(BU AS STRING)                                                                AS BU
      , CAST(Account_No AS STRING)                                                        AS Account_No
      , CAST(Type_Of_Issuing AS STRING)                                                   AS Type_Of_Issuing
      , CAST(Clean_Account_No AS STRING)                                                  AS Clean_Account_No
      , CAST(Contract AS STRING)                                                          AS Contract
      , CAST(Case_Code AS STRING)                                                         AS Case_Code
      , CAST(CIF AS STRING)                                                               AS CIF
      , CAST(CUST_AFM AS STRING)                                                          AS CUST_AFM
      , CAST(First_Name AS STRING)                                                        AS First_Name
      , CAST(Last_Name AS STRING)                                                         AS Last_Name
      , CAST(Relation_Type AS STRING)                                                     AS Relation_Type
      , CAST(Account_First_Letter AS STRING)                                              AS Account_First_Letter
      , CAST(Account_First_Letter_date AS DATE)                                           AS Account_First_Letter_date
      , CAST(`UPDATED_BY (1st_LETTER)` AS STRING)                                         AS UPDATED_BY_1st_LETTER
      , CAST(`Account First Reminder Letter` AS STRING)                                   AS Account_First_Reminder_Letter
      , CAST(`Account First Reminder letter Date` AS DATE)                                AS Account_First_Reminder_letter_Date
      , CAST(`Updated by (1st reminder letter)` AS STRING)                                AS Updated_by_1st_reminder_letter
      , CAST(Account_Second_Letter AS STRING)                                             AS Account_Second_Letter
      , CAST(Account_Second_Letter_date AS DATE)                                          AS Account_Second_Letter_date
      , CAST(`UPDATED_BY (2nd_LETTER)` AS STRING)                                         AS UPDATED_BY_2nd_Letter
      , CAST(Account_Third_Letter AS STRING)                                              AS Account_Third_Letter
      , CAST(Account_Third_Letter_date AS DATE)                                           AS Account_Third_Letter_date
      , CAST(`UPDATED_BY (3rd_LETTER)` AS STRING)                                         AS UPDATED_BY_3rd_LETTER
      , CAST(REPLACE(`CoC_Outstanding_Balance(LCY)` ,',','.') AS DECIMAL(19,2))           AS CoC_Outstanding_Balance_LCY
      , CAST(Account_Active_Application_for_Code AS STRING)                               AS Account_Active_Application_for_Code
      , CAST(Account_Status AS STRING)                                                    AS Account_Status
      , COALESCE(try_to_date(regexp_replace(Account_Status_Last_Update, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(Account_Status_Last_Update, 'yyyy-MM-dd')) AS Account_Status_Last_Update
      , CAST(`Letter_(Conduct_Code)` AS STRING)                                           AS Letter_Conduct_Code
      , COALESCE(try_to_date(regexp_replace(`Letter_Date_(Conduct_Code)`, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(`Letter_Date_(Conduct_Code)`, 'yyyy-MM-dd')) AS Letter_Date_Conduct_Code
      , CAST(Customer_Active_Application_for_Code AS STRING)                              AS Customer_Active_Application_for_Code
      , CAST(Account_Application_for_code_Decision_Completed_Ever AS STRING)              AS Account_Application_for_code_Decision_Completed_Ever
      , COALESCE(try_to_date(regexp_replace(Account_Application_for_code_Decision_Completed_Ever_Date, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(Account_Application_for_code_Decision_Completed_Ever_Date, 'yyyy-MM-dd')) AS Account_Application_for_code_Decision_Completed_Ever_Date
      , CAST(`Active Appeal` AS STRING)                                                   AS Active_Appeal
      , COALESCE(try_to_date(regexp_replace(`Date of Appeal`, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(`Date of Appeal`, 'yyyy-MM-dd')) AS Date_of_Appeal
      , CAST(`Committee Decision (LoV)` AS STRING)                                        AS Committee_Decision_LoV
      , COALESCE(try_to_date(regexp_replace(`Decision Date`, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(`Decision Date`, 'yyyy-MM-dd')) AS Decision_Date
      , CAST(`Exclude from CoC Letters` AS STRING)                                        AS Exclude_from_CoC_Letters
      , CAST(`Exclusion Reason from CoC Letters (LoV)` AS STRING)                         AS Exclusion_Reason_from_CoC_Letters_LoV
      , COALESCE(try_to_date(regexp_replace(`Coc Exclusion Date`, '\\s+', ' '), 'MMM d yyyy hh:mma'), try_to_date(`Coc Exclusion Date`, 'yyyy-MM-dd')) AS Coc_Exclusion_Date
      , CAST(getdate() AS TIMESTAMP)                                                      AS InsertTimeStamp
FROM `{env_prefix}-cdh`.landed.repointqcab_coc_fields c
LEFT JOIN NonCorePerimeter e
ON right(e.ExposureKey,len(e.ExposureKey)-3) = c.Clean_Account_No 
WHERE 1=1
AND c.Type_Of_Issuing NOT LIKE '%PRF%'
AND c.CIF NOT LIKE 'G%'
AND c.CIF NOT LIKE 'CS$'
AND c.BusinessDate = '{run_date}'
AND e.ExposureKey IS NULL 
''')
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_CocFields: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["Case_Code"]
check_table(delivery_table_path, som_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
coc_fields = spark.sql(
f'''
SELECT * 
FROM {delivery_table_path}
where BusinessDate = {som_date!r}
'''
)
coc_fields.createOrReplaceTempView("coc_fields")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/CoC_Fields/"
file_name = "CoC_Fields_" +som_date+ ".csv"
coc_fields.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)